# Milestone 2: RAG Pipeline Exploration

This notebook explores a Retrieval-Augmented Generation (RAG) pipeline
for Amazon Digital Music reviews.

We evaluate:
- Retrieval quality
- Prompt design
- Generated responses

In [4]:
import sys
import os

# Go one level up from notebooks → project root
sys.path.append(os.path.abspath(".."))

In [5]:
from src.semantic import load_faiss
from src.rag_pipeline import run_rag_pipeline, load_llm
from src.prompts import build_rag_prompt

/Users/VS-subbu/miniforge3/envs/search-app/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
vector_store = load_faiss()
llm = load_llm()

In [7]:
query = "What is a good relaxing album?"

result = run_rag_pipeline(query, vector_store=vector_store, llm=llm)

print("Answer:\n", result["answer"])

Answer:
 Impressions: First Snow, Oceanside Piano, and Wolf in Sheep's Clothing - EP are good relaxing albums, according to the reviews.


In [8]:
print("Context:\n")
print(result["context"])

Context:

[Document 1]
Product Title: Impressions: First Snow
Review Title: Quiet Pleasure
Review Text: time after time i go to this cd for relaxation, setting a quiet but light mood, or as background music for journaling or other writing; i like it so much that i gave one to my massage therapist. it has been my favorite album for close to a decade. "the stillness of white," track 5, evokes the deep, rich silence after a heavy snowfall. hear the suggestion of tinkling icicles in "field of crystal," track 3. each track takes you away and warms your heart like a first snow.
Similarity Score: 0.7105

[Document 2]
Product Title: Wolf in Sheep's Clothing - EP
Review Title: Amazing!!
Review Text: beautiful relaxing amazing music! it’s a must have
Similarity Score: 0.7447

[Document 3]
Product Title: Oceanside Piano
Review Title: Beautiful and relaxing
Review Text: this piano music is beautiful and very relaxing. i've listened to many cds while on a massage table but this is my favorite. my t

Prompt Variant 1 — Basic

In [9]:
def prompt_v1(query, context):
    return f"""
Answer the question based on the context.

Context:
{context}

Question:
{query}

Answer:
"""

Prompt Variant 2 — Strict

In [10]:
def prompt_v2(query, context):
    return f"""
You are a helpful assistant answering questions using ONLY the given context.

If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{query}

Answer clearly and concisely:
"""

Prompt Variant 3 — Structured

In [ ]:
def prompt_v3(query, context):
    return f"""
You are a music recommendation assistant.

Use the context to answer the query. If exact details (like song names)
are not present, give a helpful recommendation based on the album or artist.

Rules:
- Prefer album or artist names if song names are missing
- You may infer general recommendations
- Keep answer under 200 characters
- Do NOT say "I don't know" unless context is completely irrelevant

Context:
{context}

Question:
{query}

Answer:
"""

In [16]:
query = "Which albums are good for studying?"

base = run_rag_pipeline(query, vector_store=vector_store, llm=llm)

context = base["context"]

for i, prompt_fn in enumerate([prompt_v1, prompt_v2, prompt_v3], start=1):
    print(f"\n--- Prompt Variant {i} ---\n")
    
    prompt = prompt_fn(query, context)
    
    from src.rag_pipeline import generate_answer
    answer = generate_answer(prompt, llm)
    
    print(answer)


--- Prompt Variant 1 ---

Based on the context, it seems that only one album is explicitly mentioned as being good. 

The review for "Everyone's Beautiful" by Waterdeep states: "i mainly bought this album for the title track". However, this doesn't specifically indicate it's a good album for studying. 

However, if the goal is to find an album without negative reviews, then "Everyone's Beautiful" by Waterdeep and "Gospel Collection by Dolly Parton" could be considered options. 

However, based on the information given, it is difficult to recommend an album for studying as there is no specific information that any of the albums are suitable for studying.

--- Prompt Variant 2 ---

I don't know.

--- Prompt Variant 3 ---

Based on the context, I'd recommend calm and non-distracting albums. Considering the "Gospel Collection by Dolly Parton" is a soothing gospel album, I'd suggest:

1. Gospel Collection by Dolly Parton
2. Waterdeep's "Everyone's Beautiful" (title track is a gentle, acous

In [17]:
queries = [
    "What are relaxing albums?",
    "Best music for studying?",
    "Good workout music?",
]

for q in queries:
    print(f"\n=== Query: {q} ===\n")
    
    result = run_rag_pipeline(q, vector_store=vector_store, llm=llm)
    print(result["answer"])


=== Query: What are relaxing albums? ===

Based on the provided context, relaxing albums include:

1. Wolf in Sheep's Clothing - EP (mentioned as "beautiful relaxing amazing music")
2. Impressions: First Snow (mentioned as "for relaxation, setting a quiet but light mood")
3. Oceanside Piano (mentioned as "beautiful and very relaxing")

=== Query: Best music for studying? ===

Based on the information provided, I don't know the best music for studying. None of the review texts mention studying or provide information about the music being suitable for that purpose.

=== Query: Good workout music? ===

Yes, according to Document 1, the product has "tons of" fast workout songs.


Failure Case

In [18]:
query = "What is the best phone?"

result = run_rag_pipeline(query, vector_store=vector_store, llm=llm)

print(result["answer"])

I don't know.


## Observations

### Prompt Comparison

**Prompt Variant 1 (Basic Prompt):**
- Generates answers even when context is weak or irrelevant.
- Tends to make assumptions (e.g., inferring study suitability from morning routines).
- Higher risk of **hallucination**.
- Useful for exploratory responses but less reliable.

**Prompt Variant 2 (Strict Prompt):**
- Most **conservative and reliable**.
- Correctly outputs *"I don't know"* when the answer is not supported by context.
- Minimizes hallucinations.
- Best suited for applications where **accuracy and grounding** are important.

**Prompt Variant 3 (Structured Prompt):**
- Produces more **interpretable and structured answers**.
- Attempts to extract meaningful patterns (e.g., mellow tone → suitable for studying).
- Slightly more informative than Variant 2 but may still infer beyond explicit context.
- Good balance between informativeness and structure.

---

### Retrieval Quality

- Retrieval performs well for **domain-relevant queries** such as:
  - "What are relaxing albums?"
  - Successfully extracts albums like *Wolf in Sheep's Clothing*, *First Snow*, and *Oceanside Piano*.

- Retrieval struggles when:
  - Query intent is **not explicitly represented** in reviews (e.g., "studying", "workout").
  - Leads to either:
    - weak inference (Prompt V1, V3), or
    - correct abstention (Prompt V2).

---

### Query Difficulty Observations

- **Easy Queries:**
  - Queries directly aligned with review language (e.g., "relaxing albums").
  - Clear signal in text → strong RAG performance.

- **Medium Difficulty:**
  - Queries requiring light interpretation (e.g., "studying music").
  - Performance depends heavily on prompt design.

- **Hard Queries:**
  - Queries with **no direct evidence** in context (e.g., "workout music").
  - System either fails gracefully (V2) or produces minimal/weak output.

---

### Overall Findings

- Prompt design significantly impacts RAG performance.
- **Prompt Variant 2** is the most reliable for factual grounding.
- **Prompt Variant 3** provides better readability and structured output.
- Retrieval is effective but limited by the **semantic coverage of the dataset**.

---

### Key Takeaways

- Strong prompt constraints reduce hallucination.
- RAG systems are highly dependent on:
  - quality of retrieved documents
  - alignment between query and dataset
- Combining **good retrieval + strict prompting** yields the best results.